# AI-RAG · Week 13 — Notebook 1 (Day 1)
## Tool Calling Foundations & Tool Design

Phase 3 begins: your assistant stops being a pure RAG pipeline and becomes an **agent** — it decides which tool to call, calls it, observes the result, and answers. This notebook is the tool interface, a hand-built agent loop, the four tool-design principles, and the **tool-design clinic**. It runs **fully offline** via a deterministic `StubRouter` + fixtures (no OpenAI/Tavily key).

> Code is lifted **verbatim** from the package (`src/agents/**`, `scripts/**`). Changes are notebook glue only: local data paths, dropped relative imports (the code is inlined), removed CLI blocks — and the model string is updated to `gpt-5-mini` to match the corrected deck (that line is the real OpenAI path, not exercised offline).

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# 1) ensure pydantic  2) write the offline data files the tools read.
# The agent runs offline by default (deterministic StubRouter + fixtures) — no
# OpenAI/Tavily key needed. The SQLite side-DB is built in the next cell.
try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic"])

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/web_fixtures.json").write_text(r"""{
  "ai agents": [
    {"title": "State of AI Agents 2026", "snippet": "Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator."},
    {"title": "Agent frameworks roundup", "snippet": "Raw loops and graph-based orchestration remain the two dominant patterns."}
  ],
  "openai api": [
    {"title": "OpenAI API pricing", "snippet": "gpt-4o-mini is positioned as the low-cost tier for high-volume workloads."}
  ],
  "cricket": [
    {"title": "Cricket World Cup — result", "snippet": "The most recent tournament concluded with a closely contested final."}
  ],
  "python libraries": [
    {"title": "Trending Python libraries", "snippet": "Data, LLM-tooling, and async libraries dominate this year's trending lists."}
  ],
  "announce": [
    {"title": "OpenAI announcements", "snippet": "Recent announcements focus on tool use, structured outputs, and cost reductions."}
  ],
  "_default": [
    {"title": "Web result", "snippet": "A relevant public-web snippet would appear here (offline fixture)."}
  ]
}""" + "\n")
Path("data/tool_selection_set.jsonl").write_text(r"""{"id": "calc_1", "question": "What is 1350 * 0.18?", "expected_tool": "calculate"}
{"id": "calc_2", "question": "Calculate 4200 / 12.", "expected_tool": "calculate"}
{"id": "calc_3", "question": "How much is 15% of 890?", "expected_tool": "calculate"}
{"id": "calc_4", "question": "What is (320 + 480) * 2?", "expected_tool": "calculate"}
{"id": "calc_5", "question": "What is 900 - 275?", "expected_tool": "calculate"}
{"id": "sql_1", "question": "How many documents are in the knowledge base?", "expected_tool": "query_sql"}
{"id": "sql_2", "question": "How many HR documents do we have?", "expected_tool": "query_sql"}
{"id": "sql_3", "question": "Which department has the most documents?", "expected_tool": "query_sql"}
{"id": "sql_4", "question": "What version is the Acceptable Use Policy?", "expected_tool": "query_sql"}
{"id": "sql_5", "question": "List the Finance documents and when they were last updated.", "expected_tool": "query_sql"}
{"id": "web_1", "question": "What is the latest news on AI agents in 2026?", "expected_tool": "search_web"}
{"id": "web_2", "question": "What is the current price of the OpenAI API for gpt-4o-mini?", "expected_tool": "search_web"}
{"id": "web_3", "question": "Who won the most recent Cricket World Cup?", "expected_tool": "search_web"}
{"id": "web_4", "question": "What are the trending Python libraries this year?", "expected_tool": "search_web"}
{"id": "web_5", "question": "What did OpenAI announce today?", "expected_tool": "search_web"}""" + "\n")
print("wrote offline data files (web fixtures + tool-selection set)")


wrote offline data files (web fixtures + tool-selection set)


### Setup · the side-database (`scripts/seed_db.py`)

The SQL tool queries a small structured side-table. This builds it offline — `document_metadata` (one row per KB document) and `audit_log` (every SQL query the agent runs).

In [2]:
#!/usr/bin/env python3
"""scripts/seed_db.py — (re)create the capstone side database.

W13 gives the assistant structured data to query as a tool. This builds a small
SQLite database with two tables:

  document_metadata  — one row per document in the knowledge base (the structured
                       side-table the SQL tool queries)
  audit_log          — every SQL query the agent runs is logged here (the audit
                       rail for SQL-as-a-tool)

SQLite is local and offline, so the whole SQL-tool exercise runs with no network
and no credentials. Run this once (smoke_test.sh does it for you):

    python3 scripts/seed_db.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

DB_PATH = Path("data/capstone_side.db")   # notebook: local file

# A small, realistic slice of the Enterprise Knowledge Assistant's corpus,
# across the three departments the capstone covers.
DOCS = [
    # source, title, doc_type, department, version, updated_at, chunk_count
    ("hr-001", "Leave & Time-Off Policy",          "policy",   "HR",      "v3", "2026-01-14", 42),
    ("hr-002", "Code of Conduct",                  "policy",   "HR",      "v2", "2025-11-02", 55),
    ("hr-003", "Remote Work Guidelines",           "guideline","HR",      "v4", "2026-02-20", 28),
    ("hr-004", "Onboarding Handbook",              "handbook", "HR",      "v1", "2025-06-30", 90),
    ("it-001", "Acceptable Use Policy",            "policy",   "IT",      "v5", "2026-01-30", 37),
    ("it-002", "Security Incident Runbook",        "runbook",  "IT",      "v2", "2026-02-11", 33),
    ("it-003", "VPN & Access Setup",               "guideline","IT",      "v3", "2025-12-05", 19),
    ("it-004", "Data Classification Standard",     "standard", "IT",      "v2", "2026-01-08", 24),
    ("fin-001","Expense Reimbursement Policy",     "policy",   "Finance", "v3", "2026-01-22", 31),
    ("fin-002","Procurement Guidelines",           "guideline","Finance", "v1", "2025-09-15", 40),
    ("fin-003","Travel Policy",                    "policy",   "Finance", "v2", "2025-10-19", 26),
    ("fin-004","Corporate Card Terms",             "terms",    "Finance", "v4", "2026-02-01", 18),
]


def seed(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE document_metadata (
            id           INTEGER PRIMARY KEY,
            source       TEXT NOT NULL,
            title        TEXT NOT NULL,
            doc_type     TEXT NOT NULL,
            department   TEXT NOT NULL,
            version      TEXT NOT NULL,
            updated_at   TEXT NOT NULL,
            chunk_count  INTEGER NOT NULL
        )""")
    cur.executemany(
        "INSERT INTO document_metadata "
        "(source,title,doc_type,department,version,updated_at,chunk_count) "
        "VALUES (?,?,?,?,?,?,?)", DOCS)
    cur.execute("""
        CREATE TABLE audit_log (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            ts         TEXT NOT NULL,
            actor      TEXT NOT NULL,
            request    TEXT,
            sql        TEXT,
            decision   TEXT NOT NULL,
            row_count  INTEGER,
            error      TEXT
        )""")
    con.commit()
    con.close()
    return db_path

seed()
print('side-DB built:', DB_PATH, '· tables: document_metadata, audit_log')


side-DB built: data/capstone_side.db · tables: document_metadata, audit_log


## 1 · The tool interface — `src/agents/tools/base.py`

A tool an LLM can call reliably is four things done well — a good **name**, a typed **schema** (a Pydantic model), a **description** written for the model, and declared **idempotency**. Every tool subclasses `Tool`, sets those four, and implements `run`. One definition **exports to both** OpenAI function-calling and Anthropic tool-use schemas — the same internal `Tool`, rendered to each provider's wire format.

In [3]:
"""src/agents/tools/base.py — the Tool contract.

A tool an LLM can call reliably is four things done well (the W13 principles):
  1. a good NAME       — a verb, specific, snake_case
  2. a typed SCHEMA    — a Pydantic model: required fields, sensible defaults
  3. a DESCRIPTION     — written for the *model*, saying WHEN to use the tool
  4. IDEMPOTENCY        — declared, so the loop knows if a retry is safe

Every tool subclasses `Tool`, sets those four, and implements `run`. The same
definition exports to both OpenAI function-calling and Anthropic tool-use schemas.
"""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel


class Tool:
    name: str = ""
    description: str = ""
    Args: type[BaseModel] = BaseModel
    idempotent: bool = True          # is calling this twice with the same args safe?
    has_side_effects: bool = False   # does it change external state?

    def run(self, **kwargs: Any) -> str:
        raise NotImplementedError

    # ── validation + execution ────────────────────────────────────────────
    def call(self, arguments: dict[str, Any]) -> str:
        """Validate arguments against the schema, then run."""
        args = self.Args(**(arguments or {}))
        return self.run(**args.model_dump())

    # ── provider schema exports ───────────────────────────────────────────
    def parameters_schema(self) -> dict:
        return self.Args.model_json_schema()

    def openai_schema(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name,
                             "description": self.description,
                             "parameters": self.parameters_schema()}}

    def anthropic_schema(self) -> dict:
        return {"name": self.name,
                "description": self.description,
                "input_schema": self.parameters_schema()}


## 2 · A concrete tool — `src/agents/tools/calculate.py`

The clean example of all four principles. Instantiate it, see it render to **both** providers from one definition, and run it through the validating `call()`.

In [4]:
"""src/agents/tools/calculate.py — a computation tool.

Deterministic, offline, and safe: it evaluates an arithmetic expression with a
small AST walker (no `eval`), so an LLM can hand it a calculation and get a
reliable number back. Idempotent and side-effect-free — the easy end of the
tool spectrum, and a good first example of a well-formed tool.
"""
from __future__ import annotations

import ast
import operator

from pydantic import BaseModel, Field

# Tool is defined in the cell above

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos, ast.FloorDiv: operator.floordiv,
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression — only + - * / % ** ( ) and numbers")


class CalculateArgs(BaseModel):
    expression: str = Field(
        ..., description="An arithmetic expression, e.g. '1350 * 0.18' or '(42 + 8) / 2'.")


class CalculateTool(Tool):
    name = "calculate"
    description = ("Evaluate an arithmetic expression and return the numeric result. "
                   "Use this whenever a question needs exact math — sums, differences, "
                   "products, division, percentages, or powers. Do NOT use it to look up "
                   "facts; it only does arithmetic on numbers you provide.")
    Args = CalculateArgs
    idempotent = True
    has_side_effects = False

    def run(self, expression: str) -> str:
        try:
            result = _eval(ast.parse(expression, mode="eval"))
        except Exception as e:  # noqa: BLE001 - report cleanly to the agent
            return f"calculate error: {e}"
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"{expression} = {result}"


In [5]:
calc = CalculateTool()
print('name:', calc.name)
print('params:', list(calc.parameters_schema().get('properties', {})))
print('openai schema →', calc.openai_schema()['type'], calc.openai_schema()['function']['name'])
print('anthropic schema keys →', list(calc.anthropic_schema().keys()))
print('run "1350 * 0.18" →', calc.call({'expression': '1350 * 0.18'}))

name: calculate
params: ['expression']
openai schema → function calculate
anthropic schema keys → ['name', 'description', 'input_schema']
run "1350 * 0.18" → 1350 * 0.18 = 243


## 3 · The other two tools — `query_sql` and `search_web`

Inlined here so the agent loop can run; each gets the deep treatment (SQL safety rails, when web search earns its cost) in **Notebook 2**.

In [6]:
"""src/agents/tools/query_sql.py — SQL as a tool, safely.

Letting an LLM run SQL is powerful and dangerous. This tool ships the three rails
every SQL-as-a-tool needs in production:

  1. AUDIT LOGGING     — every query (allowed or denied) is written to audit_log
  2. READ-ONLY DEFAULT  — only a single SELECT / WITH…SELECT is permitted
  3. CONSTRAINED UPDATES — writes/DDL (INSERT/UPDATE/DELETE/DROP/…) are denied
                          outright (write access is a W18 topic, gated behind
                          injection defenses and a human-approval step)

Everything runs against a local SQLite database, so it works offline with no
credentials. The migration path to Postgres is a config change, not a rewrite
(see docs/runbooks/sql_as_a_tool_safety.md).
"""
from __future__ import annotations

import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

DB_PATH = Path("data/capstone_side.db")   # notebook: local file (built in setup)

# Statements that change data or schema — denied by the read-only rail.
_FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "replace",
              "truncate", "attach", "detach", "pragma", "vacuum", "reindex",
              "grant", "revoke", "begin", "commit"}
_MAX_ROWS = 50


def _strip_comments(sql: str) -> str:
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    return sql.strip()


def is_read_only(sql: str) -> tuple[bool, str]:
    """Return (ok, reason). Only a single SELECT/WITH…SELECT is allowed."""
    clean = _strip_comments(sql)
    if not clean:
        return False, "empty query"
    statements = [s for s in clean.split(";") if s.strip()]
    if len(statements) > 1:
        return False, "multiple statements are not allowed"
    first = statements[0].lstrip().split(None, 1)[0].lower()
    if first not in ("select", "with"):
        return False, f"only SELECT queries are allowed (got '{first.upper()}')"
    words = set(re.findall(r"[a-z_]+", clean.lower()))
    hit = words & _FORBIDDEN
    if hit:
        return False, f"write/DDL keyword not permitted: {', '.join(sorted(hit))}"
    return True, "ok"


class QuerySqlArgs(BaseModel):
    sql: str = Field(..., description="A single read-only SQL SELECT query over the "
                                      "document_metadata table.")
    purpose: str = Field("", description="One phrase describing what the query is for "
                                         "(recorded in the audit log).")


class QuerySqlTool(Tool):
    name = "query_sql"
    description = (
        "Run a read-only SQL SELECT against the knowledge-base metadata database and "
        "return the rows. Use this for questions about the DOCUMENTS THEMSELVES — how "
        "many documents exist, which department owns them, their type, version, "
        "chunk_count, or when they were last updated. Table: document_metadata"
        "(id, source, title, doc_type, department, version, updated_at, chunk_count). "
        "Do NOT use it to answer questions about document CONTENTS — only their metadata.")
    Args = QuerySqlArgs
    idempotent = True            # SELECTs don't change state
    has_side_effects = False

    def __init__(self, db_path: Path | str = DB_PATH):
        self.db_path = Path(db_path)

    def _audit(self, request: str, sql: str, decision: str,
               row_count: int | None, error: str | None) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute(
            "INSERT INTO audit_log (ts, actor, request, sql, decision, row_count, error) "
            "VALUES (?,?,?,?,?,?,?)",
            (datetime.now(timezone.utc).isoformat(timespec="seconds"), "agent",
             request, sql, decision, row_count, error))
        con.commit()
        con.close()

    def run(self, sql: str, purpose: str = "") -> str:
        ok, reason = is_read_only(sql)
        if not ok:
            self._audit(purpose, sql, "denied", None, reason)
            return (f"query_sql DENIED: {reason}. This tool is read-only; write access "
                    f"is not available. (Logged to the audit trail.)")
        try:
            con = sqlite3.connect(self.db_path)
            cur = con.execute(sql)
            rows = cur.fetchmany(_MAX_ROWS)
            cols = [d[0] for d in cur.description] if cur.description else []
            con.close()
        except Exception as e:  # noqa: BLE001
            self._audit(purpose, sql, "error", None, str(e))
            return f"query_sql error: {e}"
        self._audit(purpose, sql, "allowed", len(rows), None)
        if not rows:
            return "query_sql: (0 rows)"
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        more = "" if len(rows) < _MAX_ROWS else f"\n… (capped at {_MAX_ROWS} rows)"
        return f"{header}\n{'-' * len(header)}\n{body}{more}"


In [7]:
"""src/agents/tools/search_web.py — an external-API tool (web search).

Wraps a web-search API (Tavily) so the agent can reach information beyond the
knowledge base — current events, prices, anything not in the corpus. This is the
"API-calling agent" pattern: the LLM decides it needs fresh external info and
calls out.

Web search has real cost and latency, so the description tells the model to use
it only when the answer isn't in the corpus. For offline runs and tests, the tool
falls back to a local fixture file, so the whole thing works with no network and
no TAVILY_API_KEY.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

FIXTURES_PATH = Path("data/web_fixtures.json")   # notebook: local file (written in setup)


class SearchWebArgs(BaseModel):
    query: str = Field(..., description="A concise web-search query.")
    max_results: int = Field(3, ge=1, le=5, description="How many results to return.")


class SearchWebTool(Tool):
    name = "search_web"
    description = (
        "Search the public web and return short result snippets. Use this ONLY when the "
        "answer is not in the internal knowledge base — for current events, live figures, "
        "recent releases, or anything outside company documents. It has cost and latency, "
        "so prefer the internal tools when they can answer. Returns titles and snippets, "
        "not full pages.")
    Args = SearchWebArgs
    idempotent = True            # reading search results doesn't change state
    has_side_effects = False

    def __init__(self, offline: bool | None = None):
        # offline when explicitly set, or when there is no API key available
        self.offline = (not os.environ.get("TAVILY_API_KEY")) if offline is None else offline

    def _fixture(self, query: str, k: int) -> str:
        try:
            data = json.loads(FIXTURES_PATH.read_text())
        except Exception:
            data = {"_default": [{"title": "No fixture", "snippet": "offline stub result."}]}
        results = None
        for key, res in data.items():
            if key != "_default" and key.lower() in query.lower():
                results = res
                break
        results = (results or data.get("_default", []))[:k]
        lines = [f"[web:{query}] (offline fixture)"]
        for r in results:
            lines.append(f"- {r['title']}: {r['snippet']}")
        return "\n".join(lines)

    def run(self, query: str, max_results: int = 3) -> str:
        if self.offline:
            return self._fixture(query, max_results)
        # live path (needs network + TAVILY_API_KEY)
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
            resp = client.search(query=query, max_results=max_results)
            lines = [f"[web:{query}]"]
            for r in resp.get("results", [])[:max_results]:
                lines.append(f"- {r.get('title','')}: {r.get('content','')[:160]}")
            return "\n".join(lines)
        except Exception as e:  # noqa: BLE001
            return f"search_web error: {e} (falling back offline)\n" + self._fixture(query, max_results)


## 4 · The agent loop — `router.py` + `three_tool_agent.py`

The whole of "an agent" is one loop: **decide** (which tool?) → **call** it → **observe** → decide again → **answer**. The `StubRouter` decides deterministically by keyword offline (a test double for the *mechanics*; the real `LLMRouter` uses OpenAI function calling). The loop itself is tiny and framework-free.

In [8]:
"""src/agents/router.py — deciding which tool to call.

The router is the "brain" of the agent: given a question and the available tools,
it returns either a TOOL CALL (name + arguments) or a FINAL ANSWER.

Two implementations:

  LLMRouter  — the real thing: OpenAI function calling picks the tool. Needs
               OPENAI_API_KEY + network.
  StubRouter — a deterministic, offline stand-in that routes by keyword. It exists
               so the agent loop, the tool-selection harness, and the tests all run
               with no API key. It is NOT how you'd route in production — it's a
               test double for the *mechanics*.

A router returns a Decision: either .tool_call = (name, args) or .final = "answer".
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass

# Tool is defined in the cell above


@dataclass
class Decision:
    tool_call: tuple[str, dict] | None = None
    final: str | None = None


# ─── deterministic offline router ───────────────────────────────────────────
class StubRouter:
    """Routes by keyword; one tool call, then a final answer synthesised from the
    observation. Deterministic — same question always routes the same way."""

    def route(self, question: str, tools: list[Tool], history: list[dict]) -> Decision:
        # If a tool has already run this turn, produce the final answer.
        if any(m.get("role") == "tool" for m in history):
            obs = next(m["content"] for m in reversed(history) if m.get("role") == "tool")
            return Decision(final=f"Based on the tool result: {obs}")

        q = question.lower()
        names = {t.name for t in tools}

        # 1) arithmetic → calculate
        if "calculate" in names and re.search(r"\d\s*[-+*/%]\s*\d|percent|%|how much is", q):
            expr = self._extract_expr(question)
            return Decision(tool_call=("calculate", {"expression": expr}))

        # 2) metadata about the documents → query_sql
        sql_signals = ("how many document", "how many doc", "number of document", "which department",
                       "what version", "latest version", "list the", "count", "how many hr",
                       "how many it", "how many finance", "updated", "chunk", "doc_type", "document type")
        if "query_sql" in names and any(s in q for s in sql_signals):
            return Decision(tool_call=("query_sql", {"sql": self._canned_sql(q),
                                                     "purpose": "answer a metadata question"}))

        # 3) external / current info → search_web
        web_signals = ("latest", "current", "today", "news", "price", "release", "who won",
                       "weather", "stock", "recent", "2026", "trending", "announced")
        if "search_web" in names and any(s in q for s in web_signals):
            return Decision(tool_call=("search_web", {"query": question, "max_results": 3}))

        # default fallback: search_web if present, else the first tool
        if "search_web" in names:
            return Decision(tool_call=("search_web", {"query": question, "max_results": 3}))
        first = tools[0]
        return Decision(tool_call=(first.name, {}))

    @staticmethod
    def _extract_expr(question: str) -> str:
        m = re.search(r"[0-9][0-9\.\s\+\-\*/%\(\)]*[0-9\)]", question)
        return (m.group(0).strip() if m else "0")

    @staticmethod
    def _canned_sql(q: str) -> str:
        # A safe SELECT that satisfies the metadata question type (selection harness
        # only checks the tool choice, not the exact SQL).
        for dept in ("hr", "it", "finance"):
            if dept in q:
                return ("SELECT COUNT(*) AS n FROM document_metadata "
                        f"WHERE department = '{dept.upper() if dept != 'it' else 'IT'}'")
        if "version" in q:
            return "SELECT source, title, version FROM document_metadata ORDER BY source"
        return "SELECT COUNT(*) AS total_documents FROM document_metadata"


# ─── real OpenAI function-calling router ────────────────────────────────────
class LLMRouter:
    """Uses OpenAI function calling to choose a tool. Needs OPENAI_API_KEY."""

    def __init__(self, model: str = "gpt-5-mini"):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model
        self.system = ("You are a helpful assistant with tools. Use a tool when it can "
                       "answer the question; otherwise answer directly. Prefer internal "
                       "tools over web search when they suffice.")

    def route(self, question: str, tools: list[Tool], history: list[dict]) -> Decision:
        messages = [{"role": "system", "content": self.system},
                    {"role": "user", "content": question}, *history]
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            tools=[t.openai_schema() for t in tools], tool_choice="auto")
        msg = resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            args = json.loads(call.function.arguments or "{}")
            return Decision(tool_call=(call.function.name, args))
        return Decision(final=msg.content or "")


In [9]:
"""src/agents/three_tool_agent.py — the agent loop.

The whole of "an agent" is this loop:

    decide (which tool?) → call it → observe the result → decide again → answer

That's it. The intelligence is in the router (which tool) and the tools (what they
do); the loop just runs the cycle until the router produces a final answer or we
hit a step budget. This is the raw loop, built by hand — no framework (LangGraph
arrives in a later week).

The agent is provider-agnostic: give it any router (LLMRouter for real OpenAI
function calling, or StubRouter for deterministic offline runs) and the three
tools.
"""
from __future__ import annotations

import json
from typing import Any

# Decision, StubRouter, Tool, and the three Tool classes are defined in the cells above


def default_tools(offline: bool = True) -> list[Tool]:
    """The three W13 tools: one external API, one over structured data, one compute."""
    return [SearchWebTool(offline=offline), QuerySqlTool(), CalculateTool()]


class Agent:
    def __init__(self, tools: list[Tool], router: Any):
        self.tools = tools
        self.by_name = {t.name: t for t in tools}
        self.router = router

    def run(self, question: str, max_steps: int = 4) -> dict:
        history: list[dict] = []          # OpenAI-style assistant/tool turns
        steps: list[dict] = []
        for i in range(max_steps):
            decision: Decision = self.router.route(question, self.tools, history)
            if decision.final is not None:
                return {"question": question, "answer": decision.final, "steps": steps,
                        "tool_used": steps[-1]["tool"] if steps else None, "stopped": False}
            name, args = decision.tool_call
            tool = self.by_name.get(name)
            if tool is None:
                obs = f"error: unknown tool '{name}'"
            else:
                try:
                    obs = tool.call(args)
                except Exception as e:  # noqa: BLE001 - surface to the agent, don't crash
                    obs = f"{name} argument error: {e}"
            steps.append({"tool": name, "args": args, "observation": obs})
            call_id = f"call_{i}"
            history.append({"role": "assistant", "content": None,
                            "tool_calls": [{"id": call_id, "type": "function",
                                            "function": {"name": name,
                                                         "arguments": json.dumps(args)}}]})
            history.append({"role": "tool", "tool_call_id": call_id, "content": obs})
        return {"question": question, "answer": "(stopped: max steps reached)",
                "steps": steps, "tool_used": steps[-1]["tool"] if steps else None, "stopped": True}


def build_agent(offline: bool = True, use_llm: bool = False) -> Agent:
    """Construct the 3-tool agent. Default: fully offline with the StubRouter."""
    tools = default_tools(offline=offline)
    if use_llm:
        pass  # LLMRouter is defined above
        return Agent(tools, LLMRouter())
    return Agent(tools, StubRouter())


**Run a query offline and watch the loop pick a tool, run it, and answer:**

In [10]:
agent = build_agent(offline=True)
r = agent.run('How many HR documents are there?')
print('Q:', r['question'])
for i, s in enumerate(r['steps'], 1):
    print(f"  step {i}: {s['tool']}({s['args']}) → {s['observation'][:70]}")
print('A:', r['answer'])
print('tool_used:', r['tool_used'])

Q: How many HR documents are there?
  step 1: query_sql({'sql': "SELECT COUNT(*) AS n FROM document_metadata WHERE department = 'HR'", 'purpose': 'answer a metadata question'}) → n
-
4
A: Based on the tool result: n
-
4
tool_used: query_sql


## 5 · The four tool-design principles

When an agent picks the wrong tool, the fix is **almost never a bigger model** — it's the tool's definition. Tool design is the **dominant lever** for agent reliability:

1. **Name** — a verb, specific, `snake_case` (`query_sql`, not `data`).
2. **Schema** — a Pydantic model: required fields required, sensible defaults. (Also enable the provider's strict schema where supported, then validate business rules here.)
3. **Description** — written for the *model*, saying **when** to use the tool vs. others.
4. **Idempotency** — declared, so the loop knows whether a retry is safe.

The next section drills each principle against six broken tools.

## 6 · The tool-design clinic — `scripts/tool_clinic.py`

**The mandatory Activity.** Six broken tool definitions, each breaking exactly one principle. Read each, classify which principle it breaks, and check yourself — plus a linter you can run on any tool spec. Fully offline.

In [11]:
#!/usr/bin/env python3
"""scripts/tool_clinic.py — the W13 tool-design clinic (hands-on drill).

Tool design is ~80% of agent reliability, and most selection failures come from
badly-designed tools. This drill gives you six broken tool definitions, each
violating one of the four principles. For each: read it, decide which principle
it breaks, and check yourself. A built-in linter checks any tool against all four
principles — run it on the good reference tools and on your own rewrites.

    python3 scripts/tool_clinic.py list                 # the six broken tools
    python3 scripts/tool_clinic.py run C1               # show C1's definition
    python3 scripts/tool_clinic.py grade C1 naming      # check the principle
    python3 scripts/tool_clinic.py lint good            # lint the 3 real tools (all pass)
    python3 scripts/tool_clinic.py lint C1              # lint a broken tool (see the failure)
    python3 scripts/tool_clinic.py run-all             # used by smoke_test
    python3 scripts/tool_clinic.py answers             # answer key (instructors)

The four principles: NAMING · SCHEMA · DESCRIPTION · IDEMPOTENCY.
"""
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path


PRINCIPLES = ("naming", "schema", "description", "idempotency")

_VERBS = {"search", "query", "calculate", "compute", "get", "fetch", "list", "find",
          "lookup", "count", "create", "update", "delete", "send", "check", "read",
          "summarize", "translate", "convert", "resolve", "retrieve"}
_VAGUE = {"data", "handle", "do", "run", "process", "tool", "thing", "stuff", "manage",
          "info", "helper", "action", "execute"}
_CUES = ("use this", "use it", "use when", "only use", "returns", "do not use",
         "call this", "use only")
_OVERBROAD = ("any information", "anything the user", "all questions", "any question",
              "everything", "whatever the user")


# ─── a tool "spec" the linter understands ───────────────────────────────────
@dataclass
class ToolSpec:
    name: str
    description: str
    properties: dict          # {field: {"type": "..."}}  (JSON-schema style)
    required: list = field(default_factory=list)
    idempotent: object = None       # True / False / None (None = not declared)
    has_side_effects: bool = False


def lint(spec: ToolSpec) -> list[tuple[str, bool, str]]:
    """Check a tool spec against the four principles → [(principle, ok, detail)]."""
    out = []

    # 1 · naming
    n = spec.name
    verb = n.split("_")[0] if n else ""
    if not n or not n.replace("_", "").isalnum() or n != n.lower():
        out.append(("naming", False, f"'{n}' should be lower snake_case"))
    elif verb in _VAGUE or n in _VAGUE:
        out.append(("naming", False, f"'{n}' is vague/overloaded — start with a specific verb"))
    elif verb not in _VERBS:
        out.append(("naming", False, f"'{n}' doesn't start with a clear action verb"))
    else:
        out.append(("naming", True, f"'{n}' is a clear verb-first name"))

    # 2 · schema
    props = spec.properties or {}
    if not props:
        out.append(("schema", False, "no parameters defined"))
    else:
        untyped = [k for k, v in props.items() if not v.get("type") or v.get("type") == "any"]
        if untyped:
            out.append(("schema", False, f"untyped parameter(s): {', '.join(untyped)}"))
        else:
            out.append(("schema", True, "all parameters are typed"))

    # 3 · description
    d = (spec.description or "").lower()
    if len(d) < 40:
        out.append(("description", False, "description too short to guide tool choice"))
    elif not any(c in d for c in _CUES):
        out.append(("description", False, "no 'when to use' guidance for the model"))
    elif any(b in d for b in _OVERBROAD):
        out.append(("description", False, "over-broad — will be chosen for everything"))
    else:
        out.append(("description", True, "clear, scoped, model-facing description"))

    # 4 · idempotency
    if spec.idempotent is None:
        out.append(("idempotency", False, "idempotency not declared"))
    elif spec.has_side_effects and spec.idempotent is True:
        out.append(("idempotency", False, "has side effects but marked idempotent — retries unsafe"))
    else:
        out.append(("idempotency", True, "idempotency declared correctly"))

    return out


# ─── the six broken tools (each primarily breaks ONE principle) ─────────────
@dataclass(frozen=True)
class Case:
    id: str
    title: str
    symptom: str
    principle: str          # ANSWER KEY
    why: str
    fix: str
    spec: ToolSpec


CASES: list[Case] = [
    Case("C1", "The tool called 'data'",
         "The model almost never calls it, and when it does, for the wrong things.",
         "naming",
         "'data' is a noun, not an action, and tells the model nothing about what it does.",
         "Rename to a specific verb phrase, e.g. get_weather or fetch_exchange_rate.",
         ToolSpec("data", "Use this to get the weather for a city. Returns the current "
                          "temperature and conditions. Do not use it for forecasts.",
                  {"city": {"type": "string"}}, ["city"], idempotent=True)),
    Case("C2", "The loose schema",
         "The model passes wrong or missing arguments, and calls fail at runtime.",
         "schema",
         "Parameters are untyped, so the model can't tell what shape of input is expected.",
         "Type every parameter, mark required ones, give optionals defaults (use Pydantic).",
         ToolSpec("fetch_records", "Use this to fetch records from the orders database by "
                                   "filter. Returns matching order rows. Only for order data.",
                  {"filter": {}, "limit": {}}, [], idempotent=True)),
    Case("C3", "The silent description",
         "The model can't tell when to pick it versus another tool.",
         "description",
         "‘Gets the status.’ says nothing about WHEN to use the tool or what it returns.",
         "Write a model-facing description with a clear 'use this when…' and what it returns.",
         ToolSpec("get_status", "Gets the status.",
                  {"service": {"type": "string"}}, ["service"], idempotent=True)),
    Case("C4", "The unsafe retry",
         "On a retry after a timeout, users receive duplicate notifications.",
         "idempotency",
         "It sends a message (a side effect) but is marked idempotent, so the loop retries it "
         "freely — sending twice.",
         "Mark it non-idempotent and note the side effect; add a dedupe key before retrying.",
         ToolSpec("send_notification", "Use this to send a notification to a user by id. "
                                       "Returns a delivery receipt. Use only when asked to notify.",
                  {"user_id": {"type": "string"}, "message": {"type": "string"}},
                  ["user_id", "message"], idempotent=True, has_side_effects=True)),
    Case("C5", "The overloaded 'handle'",
         "One tool tries to do three jobs; the model picks it unpredictably.",
         "naming",
         "'handle' is vague and the tool does several things — the model can't reason about it.",
         "Split into separate, single-purpose tools, each with a specific verb name.",
         ToolSpec("handle", "Use this to handle a support request: it can look up an order, "
                            "issue a refund, or email the customer. Returns a result.",
                  {"request": {"type": "string"}}, ["request"], idempotent=True)),
    Case("C6", "The catch-all description",
         "It gets selected for almost every question, crowding out better tools.",
         "description",
         "The description claims it answers ANY question, so the model reaches for it by default.",
         "Scope the description to exactly what it does; say what NOT to use it for.",
         ToolSpec("lookup_info", "Use this to look up any information the user asks for, on any "
                                 "topic, at any time. Returns an answer.",
                  {"query": {"type": "string"}}, ["query"], idempotent=True)),
]
_BY_ID = {c.id: c for c in CASES}


# ─── good reference tools (should lint clean) ───────────────────────────────
def _good_specs() -> list[ToolSpec]:
    pass  # default_tools is defined in the cells above
    specs = []
    for t in default_tools(offline=True):
        s = t.parameters_schema()
        specs.append(ToolSpec(t.name, t.description, s.get("properties", {}),
                              s.get("required", []), idempotent=t.idempotent,
                              has_side_effects=t.has_side_effects))
    return specs


# ─── CLI ────────────────────────────────────────────────────────────────────
def _print_lint(name: str, spec: ToolSpec):
    print(f"\nlint · {name}")
    for principle, ok, detail in lint(spec):
        print(f"  {'✓' if ok else '✗'} {principle:<12} {detail}")


def cmd_list():
    print("Tool-design clinic — six broken tools. Which principle does each break?\n")
    for c in CASES:
        print(f"  {c.id}  {c.title}")
        print(f"      symptom: {c.symptom}")
    print(f"\nPrinciples: {' · '.join(PRINCIPLES)}")
    print(f"Run one:  python3 scripts/tool_clinic.py run {CASES[0].id}")


def cmd_run(cid: str):
    c = _BY_ID[cid]
    print(f"\n### {c.id} — {c.title}")
    print(f"SYMPTOM: {c.symptom}\n")
    print("TOOL DEFINITION:")
    print(f"  name:        {c.spec.name}")
    print(f"  description: {c.spec.description}")
    print(f"  parameters:  {c.spec.properties}  required={c.spec.required}")
    print(f"  idempotent:  {c.spec.idempotent}   has_side_effects: {c.spec.has_side_effects}")
    print(f"\nWhich principle does this break? "
          f"python3 scripts/tool_clinic.py grade {c.id} <{'|'.join(PRINCIPLES)}>")


def cmd_grade(cid: str, guess: str) -> int:
    c = _BY_ID[cid]
    g = (guess or "").strip().lower()
    if g not in PRINCIPLES:
        print(f"'{guess}' is not a principle. Pick: {', '.join(PRINCIPLES)}."); return 1
    if g == c.principle:
        print(f"✓ Correct — {c.principle}.\n  Why: {c.why}\n  Fix: {c.fix}"); return 0
    print(f"✗ Not {g}. Re-read the definition and try again — run the linter for a hint: "
          f"python3 scripts/tool_clinic.py lint {cid}"); return 1


def cmd_lint(target: str):
    if target == "good":
        for spec in _good_specs():
            _print_lint(spec.name + " (reference)", spec)
    elif target in _BY_ID:
        _print_lint(target, _BY_ID[target].spec)
    else:
        print(f"unknown target '{target}' — use 'good' or one of {list(_BY_ID)}")


def cmd_run_all():
    for spec in _good_specs():
        assert all(ok for _, ok, _ in lint(spec)), f"reference tool {spec.name} should lint clean"
    for c in CASES:
        principles_failed = {p for p, ok, _ in lint(c.spec) if not ok}
        assert c.principle in principles_failed, f"{c.id} should fail {c.principle}"
    print(f"  reference tools lint clean · all {len(CASES)} broken tools reproduce their flaw")


def cmd_answers():
    print("ANSWER KEY (instructors)\n")
    for c in CASES:
        print(f"{c.id} — {c.title}\n   principle: {c.principle}\n   why: {c.why}\n   fix: {c.fix}\n")


**List the six broken tools:**

In [12]:
cmd_list()

Tool-design clinic — six broken tools. Which principle does each break?

  C1  The tool called 'data'
      symptom: The model almost never calls it, and when it does, for the wrong things.
  C2  The loose schema
      symptom: The model passes wrong or missing arguments, and calls fail at runtime.
  C3  The silent description
      symptom: The model can't tell when to pick it versus another tool.
  C4  The unsafe retry
      symptom: On a retry after a timeout, users receive duplicate notifications.
  C5  The overloaded 'handle'
      symptom: One tool tries to do three jobs; the model picks it unpredictably.
  C6  The catch-all description
      symptom: It gets selected for almost every question, crowding out better tools.

Principles: naming · schema · description · idempotency
Run one:  python3 scripts/tool_clinic.py run C1


**Inspect one and classify it** (the grader confirms the principle):

In [13]:
cmd_run('C1')
_ = cmd_grade('C1', 'naming')   # try a wrong principle and re-run to test yourself


### C1 — The tool called 'data'
SYMPTOM: The model almost never calls it, and when it does, for the wrong things.

TOOL DEFINITION:
  name:        data
  description: Use this to get the weather for a city. Returns the current temperature and conditions. Do not use it for forecasts.
  parameters:  {'city': {'type': 'string'}}  required=['city']
  idempotent:  True   has_side_effects: False

Which principle does this break? python3 scripts/tool_clinic.py grade C1 <naming|schema|description|idempotency>
✓ Correct — naming.
  Why: 'data' is a noun, not an action, and tells the model nothing about what it does.
  Fix: Rename to a specific verb phrase, e.g. get_weather or fetch_exchange_rate.


**Lint the three reference tools** — a well-designed tool passes every check:

In [14]:
cmd_lint('good')


lint · search_web (reference)
  ✓ naming       'search_web' is a clear verb-first name
  ✓ schema       all parameters are typed
  ✓ description  clear, scoped, model-facing description
  ✓ idempotency  idempotency declared correctly

lint · query_sql (reference)
  ✓ naming       'query_sql' is a clear verb-first name
  ✓ schema       all parameters are typed
  ✓ description  clear, scoped, model-facing description
  ✓ idempotency  idempotency declared correctly

lint · calculate (reference)
  ✓ naming       'calculate' is a clear verb-first name
  ✓ schema       all parameters are typed
  ✓ description  clear, scoped, model-facing description
  ✓ idempotency  idempotency declared correctly


## Recap

A tool is a name + typed schema + model-facing description + declared idempotency, exported to both providers from one definition; the agent is a tiny decide→call→observe→answer loop; and **tool design — not the model — is what decides reliability**. Next, **Notebook 2** goes deep on the three tools (SQL safety rails, web-search judgment) and measures tool-selection accuracy — the new KPI #5.